In [11]:
from notebooks.presets import set_cwd
set_cwd("ts2vec")

import numpy as np

from src.data_generation.simple_data_generation import shift_time_series, shrunk_time_series

In [12]:
random_shifts = np.load("../notebooks/evaluations/data/random_shifts.npy")

In [13]:
def apply_embedding_function_shifts(data: np.ndarray, embedding_function: callable):
    all_embedding_vectors = []

    for i in range(data.shape[0]):
        embedding_vectors = [embedding_function(data[i])]

        for shift in random_shifts:
            shifted = shift_time_series(data[i], shift)
            embedding_vectors.append(embedding_function(shifted))

        all_embedding_vectors.append(embedding_vectors)
    return np.array(all_embedding_vectors)

In [14]:
random_shrunks = np.load("../notebooks/evaluations/data/random_shrunks.npy")

def apply_embedding_function_shrunk(data: np.ndarray, embedding_function: callable):
    all_embedding_vectors = []

    for i in range(data.shape[0]):
        embedding_vectors = [embedding_function(data[i])]

        for shrunk in random_shrunks:
            shifted = shrunk_time_series(data[i], shrunk)
            embedding_vectors.append(embedding_function(shifted))

        all_embedding_vectors.append(embedding_vectors)
    return np.array(all_embedding_vectors)


In [16]:
data_path = "../data/random_sequences/random_sequences.npy"
data = np.load(data_path)
data.shape

(100, 400)

In [17]:
from ts2vec import TS2Vec

model = TS2Vec(input_dims=14, device='cpu', output_dims=320)

model_path = 'training/ETTh1__my_test_run_20260218_110702/model.pkl'
model.load(model_path)

print("Model loaded successfully!")

Model loaded successfully!


In [27]:
def tf2vec_embedding_function(a: np.ndarray) -> np.ndarray:
    a = a[None, ..., None]
    a = np.repeat(a, 14, axis=2)
    v = model.encode(a, encoding_window='full_series')
    return v.reshape(-1)

In [30]:
embeddings = apply_embedding_function_shifts(data, tf2vec_embedding_function)
np.save("../notebooks/evaluations/data/ts2vec_embeddings_shifts.npy", embeddings)

In [29]:
embeddings = apply_embedding_function_shrunk(data, tf2vec_embedding_function)
np.save("../notebooks/evaluations/data/ts2vec_embeddings_shrunk.npy", embeddings)